In [ ]:
import pandas as pd
from pulp import *

In [ ]:
manvar_costs = pd.read_excel('variable_costs.xlsx', index_col = 0)

freight_costs = pd.read_excel('freight_costs.xlsx', index_col = 0)

var_cost = freight_costs/1000 + manvar_costs


fixed_costs = pd.read_excel('fixed_cost.xlsx', index_col = 0)

cap = pd.read_excel('capacity.xlsx', index_col=0)

demand = pd.read_excel('demand.xlsx', index_col=0)

In [ ]:
loc = ['USA', 'Germany', 'Japan', 'Brazil', 'India']
size = ['Low', 'High']

In [ ]:
model = LpProblem("Supply-Chain-Optimization, LpMinimize")

/usr/local/lib/python3.12/dist-packages/pulp/pulp.py:1489: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


In [ ]:
x = LpVariable.dicts("production_", [(i,j) for i in loc for j in loc],
                     lowBound=0, upBound=None, cat='continuous')
y = LpVariable.dicts("plants_",
                     [(i,s) for s in size for i in loc], cat="Binary")

In [ ]:
model += (lpSum([fixed_costs.loc[i,s] * y[(i,s)] * 1000 for s in size for i in loc])
+ lpSum([var_cost.loc[i,j] * x[(i,j)] for i in loc for j in loc]))

In [ ]:
for j in loc:
  model += lpSum([x[(i, j)] for i in loc]) == demand.loc[j, 'Demand']

for i in loc:
  model += lpSum([x[(i,j)] for j in loc]) <= lpSum([cap.loc[i,s]*y[(i,s)]*1000
                                                    for s in size])


In [ ]:
model.solve()
print("Total Costs = {:,} ($/Month)".format(int(value(model.objective))))
print('\n' + "Status: {}".format(LpStatus[model.status]))

dict_plant = {}
dict_prod = {}
for v in model.variables():
  if 'plant' in v.name:
    name = v.name.replace('plant__', '').replace('_','')
    dict_plant[name] = int(v.varValue)
    p_name = name

  else:
    name = v.name.replace('production__','').replace('_','')
    dict_prod[name] = v.varValue

  print(name, "=", v.varValue)

Total Costs = 92,981,000 ($/Month)

Status: Optimal
plants('Brazil','High') = 0.0
plants('Brazil','Low') = 1.0
plants('Germany','High') = 0.0
plants('Germany','Low') = 0.0
plants('India','High') = 1.0
plants('India','Low') = 0.0
plants('Japan','High') = 1.0
plants('Japan','Low') = 0.0
plants('USA','High') = 1.0
plants('USA','Low') = 0.0
('Brazil','Brazil') = 145000.0
('Brazil','Germany') = 0.0
('Brazil','India') = 0.0
('Brazil','Japan') = 0.0
('Brazil','USA') = 0.0
('Germany','Brazil') = 0.0
('Germany','Germany') = 0.0
('Germany','India') = 0.0
('Germany','Japan') = 0.0
('Germany','USA') = 0.0
('India','Brazil') = 0.0
('India','Germany') = 90000.0
('India','India') = 160000.0
('India','Japan') = 0.0
('India','USA') = 1500000.0
('Japan','Brazil') = 0.0
('Japan','Germany') = 0.0
('Japan','India') = 0.0
('Japan','Japan') = 1500000.0
('Japan','USA') = 0.0
('USA','Brazil') = 0.0
('USA','Germany') = 0.0
('USA','India') = 0.0
('USA','Japan') = 200000.0
('USA','USA') = 1300000.0
